In [93]:
# !pip install tensorly
# !pip install tensorly-torch

In [94]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [136]:
device = 'cuda'
times_mean = {}

batch_size = 10
n1 = 10
n2 = 10

In [96]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')

          return x

In [97]:
class TCL3(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL3, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          # x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          # x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          # x = rearrange(x, 'b w h c -> b c h w')

          return x

In [98]:
class TCL4(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL4, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          x = x.permute(0,1,3,2)
          # x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = x.permute(0,2,3,1)
          # x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = x.permute(0,3,1,2)
          # x = rearrange(x, 'b w h c -> b c h w')

          return x

In [99]:
class TCL5(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL5, self).__init__()
          #suppose it is 3d :
          # self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          # self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          # self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)
          self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True)
          self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True)
          self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True)

    def forward(self, x):
          x = torch.tensordot(x, self.w1, dims=([3],[0]))
          x = torch.tensordot(x, self.w2, dims = ([2],[0]))
          x = torch.tensordot(x, self.w3, dims = ([1],[0]))
          # x = self.fc3(x)
          # x = x.permute(0,1,3,2)
          # x = rearrange(x, 'b c h w -> b c w h')
          # x = self.fc2(x)
          # x = x.permute(0,2,3,1)
          # x = rearrange(x, 'b c w h -> b w h c')
          # x = self.fc1(x)
          # x = x.permute(0,3,1,2)
          # x = rearrange(x, 'b w h c -> b c h w')

          return x

In [100]:
class TCL6(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL6, self).__init__()
          #suppose it is 3d :
          self.layer = nn.Sequential(
              nn.Linear(input_shape[0], rank[0], bias = bias)
              ,nn.Linear(input_shape[1], rank[1], bias = bias)
              ,nn.Linear(input_shape[2], rank[2], bias = bias))


    def forward(self, x):
        x = self.layer(x)
          # x = self.fc3(x)
          # x = rearrange(x, 'b c h w -> b c w h')
          # x = self.fc2(x)
          # x = rearrange(x, 'b c w h -> b w h c')
          # x = self.fc1(x)
          # x = rearrange(x, 'b w h c -> b c h w')

        return x

In [101]:
import torch
from torch import nn
import time
import numpy as np

class TCL7(nn.Module):
    def __init__(self, input_shape, rank, bias=False):
        super(TCL7, self).__init__()
        # Efficient layer initialization
        self.weights = nn.ParameterList([
            nn.Parameter(torch.rand(rank[i], input_shape[i]).cuda()) for i in range(len(input_shape))
        ])
        self.bias = bias

    def forward(self, x):
        for i, weight in enumerate(self.weights):
            x = torch.matmul(x, weight.t())  # Efficient matrix multiplication
            if self.bias:
                x += self.bias[i]  # Add bias if needed
        return x

In [102]:
import torch
import torch.nn as nn

class TensorDotModule(nn.Module):
    def __init__(self, input_dim, output_dim, dim):
        super(TensorDotModule, self).__init__()
        self.w = nn.Parameter(torch.rand((input_dim, output_dim)), requires_grad=True)
        self.dim = dim

    def forward(self, x):
        # Perform tensor dot operation
        return torch.tensordot(x, self.w, dims=[[self.dim], [0]])

class TCL8(nn.Module):
    def __init__(self, input_shape, rank, bias=False):
        super(TCL8, self).__init__()
        self.layer = nn.Sequential(
            TensorDotModule(input_shape[0], rank[0], dim = 3),
            TensorDotModule(input_shape[1], rank[1], dim = 2),
            TensorDotModule(input_shape[2], rank[2], dim = 1)
        )

    def forward(self, x):
        x = self.layer(x)
        return x

In [103]:
from torch.cuda.amp import autocast


class TCL9(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL9, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
        with autocast():
            x = self.fc3(x)
            x = self.fc2(x)
            x = self.fc1(x)
        return x

In [104]:
class TCL10(nn.Module):
    def __init__(self, input_shape, rank, bias = False, device = device):
          super(TCL10, self).__init__()
          #suppose it is 3d :
          w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True).to(device)
          w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True).to(device)
          w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True).to(device)
          self.W = torch.kron(torch.kron(w1,w2).to(device),w3).to(device)
          self.rank = rank
          # print(self.W.shape)
          self.device = device

    def forward(self, x):
          x = torch.matmul(x.view(batch_size, 1000).to(device),self.W).to(device)
          x = x.view((batch_size,) + self.rank).to(device)

          return x

# tcl10 = TCL10((10,10,10),(10,10,10))
# temp = torch.rand((10,10,10,10))
# tcl10(temp).shape

In [137]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
    start = time.time()
    for i in range(n2):
        temp = torch.rand((batch_size,10,10,10))
        temp = temp.view(batch_size,1000)
        mlp = nn.Linear(1000,1000, False).to(device)
        temp = mlp(temp.to(device))
    end = time.time()
    times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['A single linear 1000x1000'] = np.array(times).mean()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 1.06 MiB is free. Process 14692 has 14.74 GiB memory in use. Of the allocated memory 14.62 GiB is allocated by PyTorch, and 1.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [133]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
    start = time.time()
    for i in range(n2):
        temp = torch.rand((batch_size,10))
        # temp = temp.view(batch_size,1000)
        mlp = nn.Linear(10,10, False).to(device)
        temp = mlp(temp.to(device))
    end = time.time()
    times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['A single linear 10x10'] = np.array(times).mean()

[0.21536469459533691, 0.19712543487548828, 0.20579218864440918, 0.21077299118041992, 0.20501995086669922, 0.21100211143493652, 0.20095467567443848, 0.20255565643310547, 0.20346283912658691, 0.20084023475646973]
0.20528907775878907 2.8058671498456535e-05


In [134]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
    start = time.time()
    for i in range(n2):
        temp = torch.rand((batch_size,10,10,10))
        temp = temp.view(batch_size,1000)
        W = nn.Parameter(torch.rand((1000,1000)), requires_grad=True).to(device)
        temp = torch.tensordot(temp.to(device), W, dims=([1],[0])).to(device)
    end = time.time()
    times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['A single tensordot 1000x1000'] = np.array(times).mean()

[7.67990779876709, 6.714578866958618, 7.663194417953491, 7.711283922195435, 6.702467918395996, 7.620455265045166, 6.732775926589966, 7.64779257774353, 7.67390251159668, 6.697009801864624]
7.284336900711059 0.21914984209273825


In [135]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  for i in range(n2):
      tcl10 = TCL10(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
      temp = tcl10(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with matrization and kroneker'] = np.array(times).mean()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 1.06 MiB is free. Process 14692 has 14.74 GiB memory in use. Of the allocated memory 14.62 GiB is allocated by PyTorch, and 1.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [109]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl2 = TCL2(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl2(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with rearrange'] = np.array(times).mean()

[0.31746602058410645, 0.2555558681488037, 0.23189926147460938, 0.2236802577972412, 0.22242164611816406, 0.2212660312652588, 0.2261333465576172, 0.23486876487731934, 0.2166907787322998, 0.21538162231445312]
0.2365363597869873 0.0008482915371268975


In [110]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl3 = TCL3(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl3(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['just 3 FC'] = np.array(times).mean()

[0.09992098808288574, 0.22554302215576172, 0.16862010955810547, 0.10002756118774414, 0.09963536262512207, 0.11438512802124023, 0.10528182983398438, 0.15986013412475586, 0.15300607681274414, 0.13846278190612793]
0.13647429943084716 0.0015273259684028062


In [111]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl4 = TCL4(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl4(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with permute'] = np.array(times).mean()

[0.23662567138671875, 0.27420926094055176, 0.3138308525085449, 0.30486202239990234, 0.27918243408203125, 0.19106793403625488, 0.17691683769226074, 0.19438910484313965, 0.17811870574951172, 0.17638778686523438]
0.23255906105041504 0.002815131821696468


In [112]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl5 = TCL5(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl5(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with tensordot'] = np.array(times).mean()

[0.34848713874816895, 0.16823911666870117, 0.1826460361480713, 0.163285493850708, 0.16213631629943848, 0.16139721870422363, 0.16308379173278809, 0.15761709213256836, 0.1719803810119629, 0.15811872482299805]
0.1836991310119629 0.00306712066470709


In [113]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl6 = TCL6(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl6(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with sequential 3 FC'] = np.array(times).mean()

[0.22686004638671875, 0.1293187141418457, 0.10607624053955078, 0.10534358024597168, 0.10976123809814453, 0.10825490951538086, 0.11601090431213379, 0.10374140739440918, 0.10423016548156738, 0.10381674766540527]
0.1213413953781128 0.0012930215699503832


In [114]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl7 = TCL7(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl7(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with different kernel FC'] = np.array(times).mean()

[0.10657000541687012, 0.10703730583190918, 0.09966635704040527, 0.11190009117126465, 0.21643519401550293, 0.1030886173248291, 0.10465550422668457, 0.09617376327514648, 0.09435057640075684, 0.09800314903259277]
0.1137880563735962 0.001197221831671982


In [115]:
# temp = torch.rand((batch_size,10,10,10))
# tcl8 = TCL8(input_shape = (10,10,10), rank = (5,6,7), bias = False).to(device)

# t = tcl8(temp.to(device))
# t.shape


In [116]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl8 = TCL8(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl8(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with sequential tensordot'] = np.array(times).mean()

[0.3846142292022705, 0.2393510341644287, 0.19322872161865234, 0.19166183471679688, 0.18358492851257324, 0.19163751602172852, 0.1822819709777832, 0.17377042770385742, 0.17215442657470703, 0.17674779891967773]
0.20890328884124756 0.003761095274301738


In [117]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl9 = TCL9(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  for i in range(n2):
      temp = tcl9(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with autocast 3 FC'] = np.array(times).mean()

<ipython-input-103-5b8875dac197>:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[0.3198666572570801, 0.283433198928833, 0.23792743682861328, 0.22607803344726562, 0.23429346084594727, 0.21806740760803223, 0.2117936611175537, 0.2160654067993164, 0.2148590087890625, 0.22623848915100098]
0.2388622760772705 0.0011153812347060922


In [118]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl3 = TCL3(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  scripted_model = torch.jit.script(tcl3).to(device)
  for i in range(n2):
      temp = scripted_model(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with JIT 3 FC'] = np.array(times).mean()

[0.16380596160888672, 0.3745734691619873, 0.3787558078765869, 0.13951420783996582, 0.14249157905578613, 0.1617264747619629, 0.1518692970275879, 0.15439915657043457, 0.1423025131225586, 0.16944026947021484]
0.19788787364959717 0.008078444350172163


In [119]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl5 = TCL5(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  scripted_model = torch.jit.script(tcl5).to(device)
  for i in range(n2):
      temp = scripted_model(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with JIT tensordot'] = np.array(times).mean()

[0.20758771896362305, 0.14533019065856934, 0.14363598823547363, 0.1448822021484375, 0.15098142623901367, 0.1522371768951416, 0.14026999473571777, 0.14081835746765137, 0.1381218433380127, 0.1533963680267334]
0.1517261266708374 0.00037126195038865716


In [120]:
temp = torch.rand((batch_size,10,10,10))
import time
import numpy as np

times = []
for i in range(n1):
  start = time.time()
  tcl4 = TCL4(input_shape = (10,10,10), rank = (10,10,10), bias = False).to(device)
  scripted_model = torch.jit.script(tcl4).to(device)
  for i in range(n2):
      temp = scripted_model(temp.to(device))
  end = time.time()
  times.append(end-start)

print(times)
print(np.array(times).mean(), np.array(times).var())
times_mean['with JIT permute'] = np.array(times).mean()

[0.21568942070007324, 0.3563194274902344, 0.16180419921875, 0.1699216365814209, 0.1663501262664795, 0.1657404899597168, 0.1622178554534912, 0.17256474494934082, 0.16008973121643066, 0.1621243953704834]
0.18928220272064208 0.003341043936796382


In [121]:
times_mean

{'A single linear 1000x1000': 0.05109083652496338,
 'A single linear 10x10': 0.029389357566833495,
 'A single tensordot 1000x1000': 0.051475238800048825,
 'with matrization and kroneker': 0.0500518798828125,
 'with rearrange': 0.2365363597869873,
 'just 3 FC': 0.13647429943084716,
 'with permute': 0.23255906105041504,
 'with tensordot': 0.1836991310119629,
 'with sequential 3 FC': 0.1213413953781128,
 'with different kernel FC': 0.1137880563735962,
 'with sequential tensordot': 0.20890328884124756,
 'with autocast 3 FC': 0.2388622760772705,
 'with JIT 3 FC': 0.19788787364959717,
 'with JIT tensordot': 0.1517261266708374,
 'with JIT permute': 0.18928220272064208}